In [0]:
# =============================================================================
# NOTEBOOK  : gold_fact_inventory_full
# PURPOSE   : silver.pos_transactions + silver.warehouse_inventory
#             + silver.product_master → gold.fact_inventory_full
# LAYER     : Gold (intermediate fact — feeds fact_inventory_kpis + fact_demand_trends)
# FREQUENCY : Daily
# WRITE     : Overwrite partitioned by snapshot_date (dynamic partition overwrite)
# GRAIN     : store_id + product_id + snapshot_date
#
# LOGIC:
#   pos_agg  → aggregate POS to store+product+date grain
#   wh_agg   → aggregate warehouse to product grain (sum across warehouses)
#   pm_dedup → product master current version (is_current=True)
#   join all three → compute profitability + stock KPIs
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit, get_last_successful_run_time
from utils.schema_registry import GOLD_FACT_INVENTORY_FULL_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col, round, when,
    sum as spark_sum, avg, countDistinct,
    collect_set, first, max as spark_max,
    to_date, row_number, sha2, concat_ws, broadcast
)
from pyspark.sql.window import Window
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
POS_TABLE    = cfg["tables"]["silver_pos_transactions"]
WH_TABLE     = cfg["tables"]["silver_warehouse_inventory"]
PM_TABLE     = cfg["tables"]["silver_product_master"]
TARGET_TABLE = cfg["tables"]["gold_fact_inventory_full"]
PIPELINE     = "gold_fact_inventory_full"


In [0]:
# ── Read + Compute + Write ───────────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, POS_TABLE)
 
try:
    # ── READ ──────────────────────────────────────────────────────────────────
    pos = spark.read.table(POS_TABLE)
    wh  = spark.read.table(WH_TABLE)
    pm  = spark.read.table(PM_TABLE)
 
    rows_read = pos.count()
    print(f"[READ] {rows_read} POS transaction rows")
 
    # ── POS AGGREGATION ───────────────────────────────────────────────────────
    # Aggregate to grain: store_id + product_id + snapshot_date
    # snapshot_date derived from transaction_ts (our column name)
    pos_agg = (
        pos
        .withColumn("snapshot_date", to_date(col("transaction_ts")))
        .groupBy("store_id", "product_id", "snapshot_date")
        .agg(
            spark_sum(col("quantity").cast("long")).alias("units_sold"),
            spark_sum(col("total_amount").cast("double")).alias("total_revenue"),
            avg(col("unit_price").cast("double")).alias("avg_selling_price"),
            countDistinct("transaction_id").alias("transaction_count"),
            countDistinct("customer_id").alias("unique_customers"),
            # Channel split
            spark_sum(when(col("channel") == "online",
                col("quantity")).otherwise(0)).alias("units_sold_online"),
            spark_sum(when(col("channel") == "offline",
                col("quantity")).otherwise(0)).alias("units_sold_offline"),
            # Payment method split
            spark_sum(when(col("payment_method") == "cash",
                col("total_amount")).otherwise(lit(0.0))).alias("revenue_cash"),
            spark_sum(when(col("payment_method") == "credit_card",
                col("total_amount")).otherwise(lit(0.0))).alias("revenue_credit_card"),
            spark_sum(when(col("payment_method") == "debit_card",
                col("total_amount")).otherwise(lit(0.0))).alias("revenue_debit_card"),
            spark_sum(when(col("payment_method") == "mobile_payment",
                col("total_amount")).otherwise(lit(0.0))).alias("revenue_mobile_payment"),
        )
    )
 
    # ── WAREHOUSE AGGREGATION ─────────────────────────────────────────────────
    # Aggregate to product grain — sum stock across all warehouses
    # Our silver uses current_stock / reserved_stock / available_stock
    wh_agg = (
        wh
        .groupBy("product_id")
        .agg(
            spark_sum(col("current_stock").cast("long")).alias("current_stock_qty"),
            spark_sum(col("reserved_stock").cast("long")).alias("reserved_stock_qty"),
            spark_sum(col("available_stock").cast("long")).alias("available_stock_qty"),
            spark_max(col("reorder_level").cast("long")).alias("reorder_level"),
            spark_sum(col("max_stock").cast("long")).alias("max_stock"),
            first("location_zone").alias("primary_location_zone"),
            countDistinct("warehouse_id").alias("warehouse_count"),
            collect_set("warehouse_id").alias("warehouse_ids"),
            spark_max("last_updated").alias("last_stock_updated"),
        )
    )
 
    # ── PRODUCT MASTER DEDUP ──────────────────────────────────────────────────
    # Use is_current=True from SCD2 silver — guaranteed one row per product
    pm_dedup = (
        pm
        .filter(col("is_current") == True)
        .select(
            "product_id",
            "product_name",
            "category",
            "subcategory",
            "brand",
            "supplier_id",
            col("cost_price").cast("double"),
            col("selling_price").cast("double"),
            col("weight").cast("double"),
            "status",
            col("effective_date").alias("product_effective_date"),
        )
    )
 
    # ── JOIN ──────────────────────────────────────────────────────────────────
    # POS grain joined with product master and warehouse
    # Inner join with product master — only products we know about
    # Inner join with warehouse — only products with warehouse data
    w30_cover = (
        Window.partitionBy("store_id", "product_id")
              .orderBy("snapshot_date")
              .rowsBetween(-30, 0)
    )
 
    fact_raw = (
        pos_agg
        .join(broadcast(pm_dedup), "product_id", "inner")
        .join(broadcast(wh_agg),   "product_id", "inner")
    )
 
    # ── DERIVED COLUMNS ───────────────────────────────────────────────────────
    df = (
        fact_raw
 
        # Profitability
        .withColumn("gross_profit",
            round((col("avg_selling_price") - col("cost_price")) * col("units_sold"), 2))
        .withColumn("gross_margin_pct",
            round(
                when(col("total_revenue") > 0,
                    (col("total_revenue") - col("cost_price") * col("units_sold"))
                    / col("total_revenue") * 100
                ).otherwise(0.0), 2))
        .withColumn("profit_per_unit",
            round(col("avg_selling_price") - col("cost_price"), 2))
 
        # 30-day rolling avg daily sales — needed for stock_cover_days
        .withColumn("avg_daily_sales_30d",
            round(avg("units_sold").over(w30_cover), 2))
 
        # Stock cover: days of stock left at current sales rate
        .withColumn("stock_cover_days",
            round(
                when(col("avg_daily_sales_30d") > 0,
                     col("available_stock_qty") / col("avg_daily_sales_30d"))
                .otherwise(lit(None).cast("double")), 1))
 
        # Inventory health flags
        .withColumn("reorder_flag",
            when(col("available_stock_qty") <= col("reorder_level"),
                 lit(1)).otherwise(lit(0)))
        .withColumn("stockout_flag",
            when(col("available_stock_qty") == 0,
                 lit(1)).otherwise(lit(0)))
        .withColumn("overstock_flag",
            when(col("current_stock_qty") > col("max_stock") * 0.9,
                 lit(1)).otherwise(lit(0)))
 
        # Sell-through rate
        .withColumn("sell_through_rate",
            round(
                when((col("units_sold") + col("available_stock_qty")) > 0,
                     col("units_sold") /
                     (col("units_sold") + col("available_stock_qty")) * 100
                ).otherwise(0.0), 2))
 
        # Stock utilisation
        .withColumn("stock_utilisation_pct",
            round(
                when(col("max_stock") > 0,
                     col("current_stock_qty") / col("max_stock") * 100
                ).otherwise(0.0), 1))
 
        # Online sales pct
        .withColumn("online_sales_pct",
            round(
                when(col("units_sold") > 0,
                     col("units_sold_online") / col("units_sold") * 100
                ).otherwise(0.0), 1))
 
        # Surrogate key: SHA256 of store_id + product_id + snapshot_date
        .withColumn("inventory_snapshot_key",
            sha2(concat_ws("|",
                col("store_id"),
                col("product_id"),
                col("snapshot_date").cast("string")
            ), 256))
 
        .withColumn("_gold_processed_at", current_timestamp())
        .withColumn("pipeline_run_id",    lit(run_id))
 
        .select([f.name for f in GOLD_FACT_INVENTORY_FULL_SCHEMA.fields])
    )
 
    rows_written = df.count()
 
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
 
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("snapshot_date")
        .option("overwriteSchema", "false")
        .saveAsTable(TARGET_TABLE)
    )
 
    print(f"[DONE] {rows_written} rows written to {TARGET_TABLE}")
 
    end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
              rows_read=rows_read, rows_written=rows_written)
 
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── Verify ───────────────────────────────────────────────────────────
from pyspark.sql.functions import col
 
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()).limit(1) \
    .select("status", "rows_read", "rows_written").show(truncate=False)
 
print("Row count:", spark.read.table(TARGET_TABLE).count())
spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1") \
    .select("operation", "operationMetrics").show(truncate=False)